# Can You Trust That Sample? T², Q, and the Two Ways to Be an Outlier
### 6.4 — PCA II: Diagnostics and Outliers

In 6.3 we built a PCA model and learned to *read* it — scores show which samples
group, loadings show which bands drive the grouping. But a scores plot never
tells you which samples to **trust**. Is that point on the edge of the cluster a
genuine extreme worth keeping, a contaminated sample, or a broken measurement?
Answering that is what turns PCA from a picture into a *diagnostic instrument*.

This notebook is about the two numbers that do it. Every sample has **two
independent distances** from a PCA model:

- **Hotelling's T²** — how far the sample sits *within* the model plane (an
  unusual but still explainable sample);
- **Q residual** (a.k.a. SPE) — how far the sample sits *off* the model plane
  (something the model has never seen).

We build both from scratch, set honest control limits, combine them in the
**influence plot**, and then prove the whole thing works by injecting outliers
whose type we know in advance — and grading our detection against that truth.

> **The one idea to hold onto.** There are two different ways to be an outlier,
> and they mean opposite things. A **high-T²** sample is *far along* the model —
> usually a real, valid extreme (keep it; maybe extend your calibration range). A
> **high-Q** sample is *far off* the model — a feature the model cannot explain
> (a contaminant, a spike, an instrument fault: investigate, usually discard).
> **An outlier is not automatically bad data.** T² vs Q is exactly what tells
> "interesting chemistry" apart from "broken measurement."

## 1. Setup

The standard stack plus two helpers from `scipy.stats` for the control limits.
We carry over `pca_svd()` from 6.3 (restated here so this notebook stands alone),
and again build the data from the shared simulator primitives so the ground truth
travels with it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import f as f_dist, norm

# Build the multi-component set from shared primitives, and reuse the cosmic-ray
# artifact from 4.5 to manufacture a known "bad measurement" outlier.
from simulated_data.core.axes import build_axis
from simulated_data.core.peaks import add_peaks
from simulated_data.core.artifacts import add_cosmic_rays

EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

SEED = 0
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

CLASS_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]


def pca_svd(X):
    '''PCA via the SVD of mean-centered data (from 6.3).

    Returns scores (U*S), loadings (rows of V^T), explained-variance ratio,
    the mean spectrum, and the singular values S.
    '''
    mean = X.mean(axis=0)
    U, S, Vt = np.linalg.svd(X - mean, full_matrices=False)
    return U * S, Vt, S**2 / np.sum(S**2), mean, S

## 2. Recap: the PCA model from 6.3

Same three-component mixture set as the previous lesson: 60 samples, each a blend
of pure components A, B, and C, in three classes enriched in one component. We
fit PCA and keep **three** components (the scree elbow we found in 6.3). This
clean set is our **reference model** — the definition of "normal" against which
every later sample is judged.

In [ ]:
x = build_axis(600.0, 1800.0, n_points=400)
COMPONENTS = {
    "A": [(700.0, 35.0, 1.0), (1150.0, 45.0, 0.6)],
    "B": [(950.0, 40.0, 0.9), (1500.0, 55.0, 0.7)],
    "C": [(1300.0, 38.0, 1.0), (1650.0, 50.0, 0.7)],
}
P = np.vstack([add_peaks(x, bands) for bands in COMPONENTS.values()])     # (3, n_points)


def make_concentrations(n_samples, seed):
    '''Class-structured concentrations: each class enriched (+0.5) in one
    component, all three also varying independently in [0.1, 1.0].'''
    rng = np.random.default_rng(seed)
    labels = np.repeat([0, 1, 2], n_samples // 3)
    enrich = np.zeros((n_samples, 3))
    enrich[np.arange(n_samples), labels] = 0.5
    return enrich + rng.uniform(0.1, 1.0, size=(n_samples, 3)), labels


conc_cal, labels_cal = make_concentrations(60, seed=SEED)
X_cal = conc_cal @ P + np.random.default_rng(SEED + 1).normal(0, 0.01, (60, 400))

scores, loadings, evr, mean_spectrum, S = pca_svd(X_cal)
K = 3                                              # components to keep (6.3 elbow)
print("calibration set :", X_cal.shape)
print("variance kept by 3 PCs : %.1f%%" % (evr[:K].sum() * 100))

## 3. Reconstruction error, revisited

In 6.3 we rebuilt spectra from `k` components and watched the error fall. That
same per-sample reconstruction error is the seed of everything here. For sample
$i$, the model's best guess is its projection onto the `K` retained components;
what's left over,

$$ e_i = x_i - \hat{x}_i, $$

is the part of the spectrum the model **cannot explain**. A well-modelled sample
has a residual that is just noise; a sample with an unmodelled feature has a
residual with *structure* — and that structure tells you what went wrong.

In [ ]:
def project(Xnew, loadings, mean, K):
    '''Project spectra onto the first K components; return scores_K and the
    reconstruction (mean + scores_K @ loadings_K).'''
    t = (Xnew - mean) @ loadings[:K].T
    recon = t @ loadings[:K] + mean
    return t, recon


t_cal, recon_cal = project(X_cal, loadings, mean_spectrum, K)
residual_cal = X_cal - recon_cal

# Per-sample residual energy -- the raw material for Q (next sections).
res_energy = np.sum(residual_cal**2, axis=1)
print("per-sample residual energy: min %.4f  median %.4f  max %.4f"
      % (res_energy.min(), np.median(res_energy), res_energy.max()))
print("all near the noise floor -> the model explains every calibration sample")

## 4. Two kinds of "far"

Picture the data as a cloud in spectrum-space and the PCA model as a plane
through it (here drawn as a line, for one component). A sample can be unusual in
**two geometrically different ways**:

- **Far along the plane** — it projects onto the model far from the centre, but
  it still *lies on* the plane. Its residual is small. This is **high T²**.
- **Far off the plane** — it floats above the model; its projection may be
  perfectly ordinary, but it sits far from the plane. This is **high Q**.

These are independent. A sample can be high in one, the other, both, or neither.
The cartoon below makes the distinction concrete before we compute anything.

In [ ]:
# A 2D schematic: a normal cloud along a model direction, plus one high-T2 point
# (far along the line) and one high-Q point (far off the line).
rng = np.random.default_rng(3)
cloud = rng.normal(0, [3.0, 0.4], size=(60, 2))          # elongated along x = "model"
p_t2 = np.array([8.5, 0.3])                               # far ALONG the model
p_q = np.array([0.5, 3.2])                                # far OFF the model

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.axhline(0, color="#888", lw=2, label="PCA model (the plane)")
ax.scatter(cloud[:, 0], cloud[:, 1], s=30, color="#bbb", label="normal samples")
ax.scatter(*p_t2, s=120, color="#1f77b4", edgecolor="k", zorder=5,
           label="high T² (far ALONG the model)")
ax.scatter(*p_q, s=120, color="#d62728", marker="^", edgecolor="k", zorder=5,
           label="high Q (far OFF the model)")
ax.annotate("", (p_q[0], p_q[1]), (p_q[0], 0),
            arrowprops=dict(arrowstyle="<->", color="#d62728"))
ax.annotate("Q = distance to model", (p_q[0] + 0.2, p_q[1] / 2), color="#d62728")
ax.annotate("", (p_t2[0], -0.0), (0, 0),
            arrowprops=dict(arrowstyle="<->", color="#1f77b4"))
ax.annotate("T² = distance within model", (2.0, -0.8), color="#1f77b4")
ax.set_xlabel("along the model"); ax.set_ylabel("off the model")
ax.set_title("Two independent ways to be an outlier")
ax.legend(loc="upper left", fontsize=8); ax.set_ylim(-1.5, 4)
fig.tight_layout(); fig.savefig(EXPORTS / "01_two_distances.png", dpi=150)
plt.show()

## 5. Hotelling's T²: distance *within* the model

T² measures how far a sample's **scores** lie from the centre, scaling each
component by how much that component normally varies (its variance $\lambda_a$):

$$ T^2_i = \sum_{a=1}^{K} \frac{t_{ia}^2}{\lambda_a}. $$

Dividing by $\lambda_a$ is the key idea — a big swing along a high-variance
component is ordinary, but the same swing along a low-variance component is
strange. T² is in units of "standard deviations of the model," so a principled
**control limit** comes from the F-distribution:

$$ T^2_{\text{lim}} = \frac{K(n-1)}{n-K}\, F_{\alpha;\,K,\,n-K}. $$

In [ ]:
n = len(X_cal)
eigenvalues = S**2 / (n - 1)                 # variance captured by each component

def hotelling_t2(scores_K, eigenvalues, K):
    '''Hotelling's T^2 per sample from the first K scores.'''
    return np.sum(scores_K[:, :K]**2 / eigenvalues[:K], axis=1)

ALPHA = 0.99
T2_cal = hotelling_t2(t_cal, eigenvalues, K)
T2_limit = K * (n - 1) / (n - K) * f_dist.ppf(ALPHA, K, n - K)
print("T² 99%% control limit : %.2f" % T2_limit)
print("calibration samples above the T² limit : %d of %d" % ((T2_cal > T2_limit).sum(), n))

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(np.arange(n), T2_cal, color="#4c72b0")
ax.axhline(T2_limit, color="red", ls="--", label=f"99% limit = {T2_limit:.1f}")
ax.set_xlabel("calibration sample"); ax.set_ylabel("Hotelling's T²")
ax.set_title("T²: distance within the model (all calibration samples in control)")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "02_hotelling_t2.png", dpi=150)
plt.show()

## 6. Q residuals (SPE): distance *to* the model

Q is simply the size of the leftover residual from Section 3 — the squared length
of what the model couldn't fit:

$$ Q_i = \sum_j e_{ij}^2 = \lVert x_i - \hat{x}_i \rVert^2. $$

A small Q means the sample lives on the model plane (it is *describable* by the
components). A large Q means the sample has a feature pointing **off** the plane.
Its control limit follows the Jackson–Mudholkar approximation, built from the
variances of the components we **discarded** (the noise subspace).

In [ ]:
def q_residual(Xnew, loadings, mean, K):
    '''Q (squared prediction error) per sample after K components.'''
    _, recon = project(Xnew, loadings, mean, K)
    return np.sum((Xnew - recon)**2, axis=1)


def q_limit(eigenvalues, K, alpha=0.99):
    '''Jackson-Mudholkar 99% control limit for Q from the discarded eigenvalues.'''
    res = eigenvalues[K:]                       # variances of dropped components
    th1, th2, th3 = res.sum(), np.sum(res**2), np.sum(res**3)
    h0 = 1 - 2 * th1 * th3 / (3 * th2**2)
    za = norm.ppf(alpha)
    return th1 * (za * np.sqrt(2 * th2 * h0**2) / th1
                  + 1 + th2 * h0 * (h0 - 1) / th1**2) ** (1 / h0)


Q_cal = q_residual(X_cal, loadings, mean_spectrum, K)
Q_limit = q_limit(eigenvalues, K, ALPHA)
print("Q 99%% control limit : %.4f" % Q_limit)
print("calibration samples above the Q limit : %d of %d" % ((Q_cal > Q_limit).sum(), n))

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar(np.arange(n), Q_cal, color="#55a868")
ax.axhline(Q_limit, color="red", ls="--", label=f"99% limit = {Q_limit:.3f}")
ax.set_xlabel("calibration sample"); ax.set_ylabel("Q residual (SPE)")
ax.set_title("Q: distance to the model (all calibration samples in control)")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "03_q_residuals.png", dpi=150)
plt.show()

## 7. The influence plot: T² and Q together

Plot T² against Q with both limits drawn and you get the single most useful
diagnostic in chemometrics — the **influence plot**. It splits samples into four
quadrants:

| | low Q | high Q |
|---|---|---|
| **low T²** | normal | off-model feature (contaminant / artifact) |
| **high T²** | valid extreme (fits, but unusual) | extreme *and* off-model |

On our clean calibration set everything sits in the safe corner — as it must,
because this *is* our definition of normal.

In [ ]:
def influence_plot(ax, T2, Q, T2_lim, Q_lim, **kw):
    ax.axvline(T2_lim, color="red", ls="--", lw=1)
    ax.axhline(Q_lim, color="red", ls="--", lw=1)
    sc = ax.scatter(T2, Q, **kw)
    ax.set_xlabel("Hotelling's T²  (distance within model)")
    ax.set_ylabel("Q residual  (distance to model)")
    return sc


fig, ax = plt.subplots(figsize=(7.5, 5.5))
influence_plot(ax, T2_cal, Q_cal, T2_limit, Q_limit, s=40, color="#888",
               label="calibration samples")
ax.set_title("Influence plot — clean calibration set (all in control)")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "04_influence_clean.png", dpi=150)
plt.show()

## 8. Putting it to the test: outliers we planted on purpose

Now we screen a fresh batch of samples through the **fixed** calibration model
(we do *not* refit — that is exactly how monitoring works: build on good data,
then judge new data against it). The batch is 15 ordinary new samples plus three
kinds of outlier, and **we know which is which**:

- **Type A — a real extreme:** a valid sample with a very high concentration of
  component A (well beyond the calibration range). It still obeys the chemistry,
  so it should be **high T², low Q**.
- **Type B — an unmodelled contaminant:** an ordinary mixture plus an extra band
  at 1000 cm⁻¹ that belongs to no calibration component. New direction → **high Q**.
- **Type C — a bad measurement:** an ordinary mixture with a **cosmic-ray spike**
  (the artifact from 4.5). Sharp, off-model → **high Q**.

The ground-truth label of every batch sample is recorded so we can grade the
diagnosis.

In [ ]:
rng = np.random.default_rng(123)

# 15 ordinary new samples (same generator as calibration).
conc_new, _ = make_concentrations(15, seed=123)
X_normal = conc_new @ P + rng.normal(0, 0.01, (15, 400))

# Type A: a genuine extreme -- very high component A (calibration maxed near 1.5).
X_A = np.array([[2.2, 0.3, 0.3]]) @ P + rng.normal(0, 0.01, (1, 400))

# Type B: ordinary mixtures + an unmodelled contaminant band at 1000 cm-1.
conc_B, _ = make_concentrations(3, seed=55)
contaminant = add_peaks(x, [(1000.0, 30.0, 0.6)])
X_B = conc_B @ P + contaminant + rng.normal(0, 0.01, (3, 400))

# Type C: ordinary mixtures + a cosmic-ray spike (a bad measurement).
conc_C, _ = make_concentrations(3, seed=77)
X_C = conc_C @ P + rng.normal(0, 0.01, (3, 400))
for i in range(3):
    X_C[i], _ = add_cosmic_rays(X_C[i], n_spikes=1, intensity=4.0, rng=rng)

X_batch = np.vstack([X_normal, X_A, X_B, X_C])
truth = np.array(["normal"] * 15 + ["A: extreme"] + ["B: contaminant"] * 3 + ["C: spike"] * 3)
print("batch:", X_batch.shape[0], "samples ->",
      dict(zip(*np.unique(truth, return_counts=True))))

In [ ]:
# Project the batch onto the FIXED calibration model and score each sample.
t_batch, _ = project(X_batch, loadings, mean_spectrum, K)
T2_batch = hotelling_t2(t_batch, eigenvalues, K)
Q_batch = q_residual(X_batch, loadings, mean_spectrum, K)

styles = {"normal": ("#888", "o"), "A: extreme": ("#1f77b4", "s"),
          "B: contaminant": ("#ff7f0e", "^"), "C: spike": ("#d62728", "D")}

fig, ax = plt.subplots(figsize=(8, 6))
ax.axvline(T2_limit, color="red", ls="--", lw=1)
ax.axhline(Q_limit, color="red", ls="--", lw=1)
for name, (c, m) in styles.items():
    sel = truth == name
    ax.scatter(T2_batch[sel], Q_batch[sel], c=c, marker=m, s=70,
               edgecolor="k", label=name)
ax.set_yscale("log")                       # spikes have enormous Q -> log axis
ax.set_xlabel("Hotelling's T²"); ax.set_ylabel("Q residual (log scale)")
ax.set_title("Influence plot with planted outliers — each type lands where predicted")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "05_influence_outliers.png", dpi=150)
plt.show()

In [ ]:
# Grade the diagnosis against the known labels.
flag_T2 = T2_batch > T2_limit
flag_Q = Q_batch > Q_limit
flagged = flag_T2 | flag_Q
is_outlier = truth != "normal"

tp = int((flagged & is_outlier).sum())
fp = int((flagged & ~is_outlier).sum())
fn = int((~flagged & is_outlier).sum())
print("DETECTION  precision = %.2f   recall = %.2f   (TP=%d FP=%d FN=%d)"
      % (tp / (tp + fp), tp / (tp + fn), tp, fp, fn))

print("\nquadrant diagnosis per planted outlier:")
print("  %-16s %8s %10s   %s" % ("type", "T²", "Q", "verdict"))
for i in np.where(is_outlier)[0]:
    verdict = ("high T² + high Q" if flag_T2[i] and flag_Q[i]
               else "high T² only" if flag_T2[i]
               else "high Q only" if flag_Q[i] else "in control")
    print("  %-16s %8.2f %10.3f   %s" % (truth[i], T2_batch[i], Q_batch[i], verdict))

Every planted outlier is caught (precision and recall both 1.0, no false alarms
on the ordinary samples), and each lands in the quadrant we predicted: the
**extreme** sample breaches **T²** while staying under the Q limit (it still fits
the chemistry), while the **contaminant** and the **spike** breach **Q** by a
wide margin (the spike's Q is orders of magnitude larger — hence the log axis).
T² and Q have *diagnosed*, not just flagged.

## 9. Why not all outliers are bad data

The influence plot does more than flag — it tells you *what to do*. The decisive
evidence is the **residual spectrum** ($x_i - \hat{x}_i$): the part the model
couldn't fit. Plot it for one of each outlier type.

- The **extreme** sample's residual is flat noise — the model explains it
  perfectly, it is simply far along a real axis. **Keep it** (and consider
  extending your calibration range to cover it).
- The **contaminant** residual shows a clean peak at 1000 cm⁻¹ — the model is
  literally pointing at the foreign band. **Investigate** (what is that?).
- The **spike** residual is a single sharp pixel — a classic bad measurement.
  **Despike or discard.**

A high-Q residual is not a verdict of "bad"; it is a *lead*. Reading it is how
you decide.

In [ ]:
# One representative residual spectrum per type, against the fixed model.
examples = {"A: extreme": 15, "B: contaminant": 16, "C: spike": 19}
_, recon_batch = project(X_batch, loadings, mean_spectrum, K)
res_batch = X_batch - recon_batch

fig, axes = plt.subplots(3, 1, figsize=(10, 7.5), sharex=True)
for ax, (name, i) in zip(axes, examples.items()):
    color = styles[name][0]
    ax.plot(x, res_batch[i], color=color, lw=1.0)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_ylabel("residual")
    ax.set_title(f"{name}  (Q = {Q_batch[i]:.3f})", loc="left", fontsize=10, color=color)
axes[0].annotate("flat noise → fits the model\n(keep it)", (1500, res_batch[15].max()*0.6),
                 fontsize=9, color="#1f77b4")
axes[1].annotate("foreign band at 1000 cm⁻¹\n(investigate)", (1010, res_batch[16].max()*0.7),
                 fontsize=9, color="#ff7f0e")
axes[2].annotate("single sharp pixel\n(bad measurement)", (1700, res_batch[19].max()*0.6),
                 fontsize=9, color="#d62728")
axes[-1].set_xlabel("Raman shift (cm$^{-1}$)")
fig.tight_layout(); fig.savefig(EXPORTS / "06_residual_spectra.png", dpi=150)
plt.show()

## 10. Leverage: which samples pull the model

A close cousin of T² is **leverage** — how much influence a calibration sample
has on the model itself. In score space it is the diagonal of the "hat" matrix,

$$ h_i = \frac{1}{n} + \sum_{a=1}^{K} \frac{t_{ia}^2}{(n-1)\lambda_a}, $$

which is just a rescaling of T² (so high-T² calibration samples are high-leverage
points). They are not necessarily wrong, but they *steer* the model, so a single
mislabelled or contaminated high-leverage sample does outsized damage. This is
why screening (this whole notebook) belongs **before** calibration in 6.5.

In [ ]:
leverage = 1.0 / n + np.sum(t_cal[:, :K]**2 / ((n - 1) * eigenvalues[:K]), axis=1)
high_lev = np.argsort(leverage)[-3:][::-1]
print("mean leverage = K/n = %.3f" % (K / n))
print("highest-leverage calibration samples (rows):", high_lev)
print("their leverage values:", np.round(leverage[high_lev], 3))
print("-> these points most steer the model; a bad one here is most dangerous.")

## 11. A reusable diagnostics helper

Bundle the whole workflow into one function: fit (or accept) a model, then return
T², Q, leverage, and the control limits for any set of spectra — ready to carry
into 6.5 (screen the calibration set before PLS) and 6.6 (monitor validation
samples).

In [ ]:
def pca_diagnostics(X_fit, K=3, alpha=0.99, X_new=None):
    '''Fit a PCA model on X_fit and return T²/Q diagnostics + limits.

    If X_new is given, the diagnostics are computed for those samples against the
    model fitted on X_fit (the monitoring use-case); otherwise they are computed
    for X_fit itself.
    '''
    scores, loadings, evr, mean, S = pca_svd(X_fit)
    n = len(X_fit)
    eig = S**2 / (n - 1)
    target = X_fit if X_new is None else X_new
    t, recon = project(target, loadings, mean, K)
    return {
        "T2": hotelling_t2(t, eig, K),
        "Q": np.sum((target - recon)**2, axis=1),
        "T2_limit": K * (n - 1) / (n - K) * f_dist.ppf(alpha, K, n - K),
        "Q_limit": q_limit(eig, K, alpha),
    }


d = pca_diagnostics(X_cal, K=3, X_new=X_batch)
print("flagged in batch:", int(((d["T2"] > d["T2_limit"]) | (d["Q"] > d["Q_limit"])).sum()),
      "of", len(X_batch), "(expected 7 planted outliers)")

## Key Takeaways

- **A PCA model has two distances, not one.** T² measures distance *within* the
  model (how extreme along the real components); Q measures distance *to* the
  model (how much the sample can't be explained). They are independent.
- **The influence plot (T² vs Q) is the master diagnostic.** Its four quadrants
  separate normal, valid-extreme, and off-model samples at a glance.
- **High T² ≠ high Q.** A valid extreme breaches T² but fits the model (low Q);
  a contaminant or artifact breaches Q while looking ordinary along the model.
- **Not all outliers are bad data.** High-T²/low-Q samples are often real and
  worth keeping (extend your range); high-Q samples are leads to investigate.
- **The residual spectrum tells you *why*.** A high-Q sample's residual points
  straight at the unmodelled feature — a foreign band, a spike, a baseline.
- **Control limits make it objective.** T² from the F-distribution, Q from
  Jackson–Mudholkar — or empirical percentiles of a known-clean set.
- **Leverage warns you which samples steer the model**, which is why you screen
  before you calibrate (6.5).
- **Ground truth proved it.** Planted outliers of known type were detected with
  perfect precision/recall and landed in the predicted quadrants.

## Practical Checklist

1. **Fit the model on known-good data**, then screen new samples against it — do
   not let suspect samples define "normal."
2. **Always compute both T² and Q**, never just a scores plot.
3. **Set control limits** (F-distribution for T², Jackson–Mudholkar for Q, or
   percentiles of a clean reference) and draw them on the influence plot.
4. **Read the residual spectrum** of every high-Q sample to identify the cause.
5. **Decide per quadrant:** high-T²/low-Q → likely a real extreme (keep, maybe
   extend range); high-Q → investigate, then despike/exclude if it is bad data.
6. **Check leverage** on the calibration set; verify high-leverage samples are
   trustworthy, since they steer the model.
7. **Re-screen after cleaning** and document which samples you removed and why.

## Common Mistakes

- **Stopping at the scores plot.** It shows T²-type structure only; an off-model
  sample (high Q) can sit comfortably inside the scores cloud and be missed.
- **Refitting the model on contaminated data.** A strong artifact becomes its own
  component, so the bad sample then looks "normal" (low Q) — always screen
  against a clean reference model.
- **Treating every outlier as bad data.** Deleting valid extremes shrinks your
  calibration range and biases later models.
- **Ignoring the residual spectrum.** A bare Q value flags a problem; the
  residual *diagnoses* it. Skipping it throws away the diagnosis.
- **Using the wrong number of components.** Too few PCs inflate Q (real signal
  leaks into the residual); too many bury genuine off-model features.
- **Forgetting limits are probabilistic.** At 99%, ~1 in 100 good samples trips a
  limit by chance — confirm before discarding.

## Reporting Guidance

- **State the model** the diagnostics use: number of components, the data it was
  fitted on, and the preprocessing applied first.
- **Report both limits** (T² and Q), the confidence level, and how they were
  derived (F-distribution / Jackson–Mudholkar / empirical percentile).
- **Show the influence plot** and identify flagged samples with their quadrant.
- **Justify every exclusion** with its residual spectrum and a reason (foreign
  band, spike, baseline) — never delete a sample on a bare number.
- **Distinguish valid extremes from bad data** explicitly; note any calibration
  range extensions prompted by high-T² samples.
- **Report how many samples were removed** and re-state the model diagnostics
  after cleaning.

## Next Lesson

**6.5 — PLS Regression: Quantitative Prediction from Spectra.** PCA and its
diagnostics describe and *screen* your data; the next step is to **predict** a
property from it. With a clean, outlier-checked calibration set in hand, PLS finds
the directions of the spectrum that co-vary with a target concentration, and we
grade its predictions against known values — with honest cross-validation to
choose the model and catch overfitting. The T²/Q habit you built here is the
quality gate that keeps those predictions trustworthy.